In [1]:
import pandas as pd
import sqlite3

In [2]:
#โหลดข้อมูล
df_raw = pd.read_csv('/content/raw_ecommerce_data.csv')

In [3]:
#ตรวจสอบโครงสร้าง
df_raw.info()
df_raw.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Order_ID       185 non-null    object
 1   Customer_Name  183 non-null    object
 2   Email          184 non-null    object
 3   Product        185 non-null    object
 4   Category       184 non-null    object
 5   Order_Date     185 non-null    object
 6   Quantity       185 non-null    int64 
 7   Unit_Price     185 non-null    object
 8   Amount         142 non-null    object
dtypes: int64(1), object(8)
memory usage: 13.1+ KB


,Order_ID,Customer_Name,Email,Product,Category,Order_Date,Quantity,Unit_Price,Amount
0,ORD-0036,Emma Brown,emma.brown@email.com,Monitor 24 inch,Electronics,24/04/2026,1,4131.00,"4,131.00"
1,ORD-0076,linda park,linda.park@email.com,Mechanical Keyboard,Electronics,"Apr 03, 2026",5,1669.50,8347.50
2,ORD-0101,john doe,john@email.com,Wireless Mouse,Electronics,11/04/2026,3,405.00,NaN
3,ORD-0059,Jane Smith,jane@email.com,Wireless Mouse,ELECTRONICS,13/03/2026,2,405.00,฿810.00
4,ORD-0038,PETER KIM,peter.kim@email.com,Gel Pen Set,Stationery,2026-02-28,5,85.50,427.50


In [5]:
#สร้าง dim_customer
dim_customer = df_raw[['Customer_Name','Email']].drop_duplicates()
# ลบ Missing Value
dim_customer = dim_customer.dropna(subset=['Customer_Name', 'Email'])
# ลบข้อมูลซ้ำ
dim_customer = dim_customer.drop_duplicates()

In [6]:
#สร้าง Surrogate key(customer_id)
dim_customer = dim_customer.reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1

In [7]:
#จัดเรียงคอมลัมน์ให้ Pk อยุ่หน้าสุด
dim_customer = dim_customer[['customer_id','Customer_Name','Email']]

In [8]:
#สร้างFact table
#นำ customer_id กลับไปใส่ใน Fact table ผ่านการ join
fact_sales = pd.merge(df_raw, dim_customer,on = ['Customer_Name','Email'],how='left')

In [9]:
#ลบคอลัมน์ text ทิ้ง เหลือไว้เพียง Foreign Key
fact_sales = fact_sales.drop(columns=['Customer_Name','Email'])

In [10]:
# สร้าง dim_product จาก Product + Category ที่ไม่ซ้ำกัน
dim_product = df_raw[['Product','Category']].drop_duplicates().reset_index(drop=True)
dim_product['Category'] = dim_product['Category'].str.strip().str.title()  # กันปัญหา ELECTRONICS vs Electronics
dim_product['product_id'] = dim_product.index + 1

In [11]:
# join กลับเข้า fact_sales แล้วดรอปคอลัมน์ text ทิ้ง
fact_sales = pd.merge(fact_sales, dim_product, on=['Product','Category'], how='left')
fact_sales = fact_sales.drop(columns=['Product','Category'])

In [12]:
# แปลง Order_Date ที่รูปแบบไม่ตรงกันให้เป็น datetime ก่อน
df_raw['Order_Date_clean'] = pd.to_datetime(df_raw['Order_Date'], format='mixed', dayfirst=True)


In [13]:
# สร้าง dim_time จากวันที่ไม่ซ้ำกัน
dim_time = df_raw[['Order_Date_clean']].drop_duplicates().reset_index(drop=True)
dim_time = dim_time.rename(columns={'Order_Date_clean':'date'})
dim_time['time_id'] = dim_time.index + 1
dim_time['year']    = dim_time['date'].dt.year
dim_time['month']   = dim_time['date'].dt.month
dim_time['day']     = dim_time['date'].dt.day
dim_time['quarter'] = dim_time['date'].dt.quarter

In [14]:
# join กลับเข้า fact_sales
fact_sales = pd.merge(fact_sales, df_raw[['Order_ID','Order_Date_clean']], on='Order_ID', how='left')
fact_sales = pd.merge(fact_sales, dim_time, left_on='Order_Date_clean', right_on='date', how='left')
fact_sales = fact_sales.drop(columns=['Order_Date_clean','date'])

In [15]:
#สร้าง connection ไปที่ file db.
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

In [16]:
#สร้าง dimention พร้อม กำหนก primary key
# 3. สร้างตาราง dim_product
cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_product (
        product_id INTEGER PRIMARY KEY,
        Product TEXT,
        Category TEXT
    )
''')

In [17]:
# 4. สร้างตาราง dim_time
cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_time (
        time_id INTEGER PRIMARY KEY,
        date TEXT,
        year INTEGER,
        month INTEGER,
        day INTEGER,
        quarter INTEGER
    )
''')

In [18]:
# 5. สร้างตาราง fact_sales พร้อม Foreign Key อ้างอิงไปยัง dimension ทั้งหมด
cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_sales (
        Order_ID TEXT PRIMARY KEY,
        customer_id INTEGER,
        product_id INTEGER,
        time_id INTEGER,
        Quantity INTEGER,
        Unit_Price REAL,
        Amount REAL,
        FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
        FOREIGN KEY (product_id) REFERENCES dim_product(product_id),
        FOREIGN KEY (time_id) REFERENCES dim_time(time_id)
    )
''')
conn.commit()

In [19]:
# เปิดใช้งานการตรวจสอบ Foreign Key ใน SQLite
cursor.execute('PRAGMA foreign_keys = ON;')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_sales (
        Order_ID TEXT PRIMARY KEY,
        customer_id INTEGER,
        product_id INTEGER,
        time_id INTEGER,
        Quantity INTEGER,
        Unit_Price REAL,
        Amount REAL,
        FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
        FOREIGN KEY (product_id) REFERENCES dim_product(product_id),
        FOREIGN KEY (time_id) REFERENCES dim_time(time_id)
    )
''')
conn.commit()

In [22]:
# โหลดข้อมูล Dimension
dim_customer.to_sql('dim_customer', con=conn,
                     if_exists='replace', index=False)

dim_product.to_sql('dim_product', con=conn,
                    if_exists='replace', index=False)

dim_time.to_sql('dim_time', con=conn,
                 if_exists='replace', index=False)

92

In [24]:
# โหลดข้อมูล Fact
fact_sales.to_sql('fact_sales', con=conn,
                   if_exists='replace', index=False)

print('ETL Pipeline ran successfully!')

ETL Pipeline ran successfully!


In [25]:
query = '''
SELECT
    c.Customer_Name,
    SUM(f.Amount) as Total_Spend
FROM fact_sales f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.Customer_Name
ORDER BY Total_Spend DESC
LIMIT 3;
'''

result = pd.read_sql_query(query, conn)
print(result)

  Customer_Name  Total_Spend
0     Ploy Kaew      57687.0
1     Narin Dee      37996.5
2      Krit Som      27846.0
